# Automated table maintenance with AIStor Tables

Streaming ingest and CDC jobs commit small batches often. Every commit to an
Apache Iceberg table adds a data file, manifests and a snapshot, so a few hours
of streaming leave a table with:

- **many small files**, which slow down every query that has to open them
- **thousands of snapshots**, which make the metadata grow without limit
- **unreferenced files**, left behind once nothing points to them anymore

Usually someone schedules Spark jobs to fix this (`rewrite_data_files`,
`expire_snapshots`, `remove_orphan_files`). **AIStor Tables does it itself.**
You set a policy once per warehouse and AIStor runs three background jobs:

| Job | What it does |
|---|---|
| Compaction | Rewrites small data files into fewer, larger ones |
| Snapshot expiration | Expires snapshots past a maximum age, always keeping a minimum number |
| Unreferenced file removal | Removes files no snapshot references, in two stages so mistakes are recoverable |

This notebook makes a messy table, has an admin turn on maintenance, and
watches AIStor clean the table up.

> **Before you start:** run `./scripts/run.sh start` on a fresh stack (or
> `./scripts/run.sh down` first). This notebook runs as `ingest-svc`, a user
> that can write tables but can't change maintenance settings.

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.compute as pc
import urllib.request

import lakehouse as lh
from make_mess import batch

cat = lh.catalog()
print(f"Connected to AIStor Tables at {lh.AISTOR_URL}/_iceberg, warehouse '{lh.WAREHOUSE}'")

## 1. Make a mess: 200 micro-batches

Each `append` is its own commit, the way a streaming job writes. 200 commits of
500 rows each give 200 small Parquet files and 200 snapshots.

In [ ]:
from datetime import datetime, timedelta, timezone

table = lh.ensure_table(cat)
BATCHES, ROWS = 200, 500
start_id = lh.health(cat, query_runs=0).rows if table.current_snapshot() else 0
t0 = datetime.now(timezone.utc)

for i in range(BATCHES):
    # lh.append retries if the commit loses to a concurrent commit (e.g. compaction)
    table = lh.append(cat, table, batch(start_id + i * ROWS, ROWS, t0 + timedelta(seconds=5 * i)))
    if (i + 1) % 50 == 0:
        print(f"committed {i + 1}/{BATCHES} batches")

## 2. Look at the mess

`lh.health()` counts snapshots and data files through the catalog, counts every
object under the table's location on storage, and times an aggregate query in
DuckDB through the same AIStor Tables catalog.

In [ ]:
before = lh.health(cat)
lh.print_health(before, "Before maintenance")

In [ ]:
table = cat.load_table(lh.IDENTIFIER)
files = table.inspect.files().to_pandas()
print(f"{len(files)} data files, median {files.file_size_in_bytes.median() / 1e3:.1f} KB")
files.file_size_in_bytes.div(1e3).describe().round(1).to_frame("file size (KB)").T

## 3. Turn on automated maintenance (admin)

Maintenance is set by an AIStor admin, not by the writer. In a terminal, run:

```bash
./scripts/run.sh policy
```

That sets one warehouse-wide policy, which every table in `lakehouse` inherits:

```bash
mc table warehouse maintenance set aistor/lakehouse --type icebergCompaction \
   --target-file-size-mb 128 --interval 1
mc table warehouse maintenance set aistor/lakehouse --type icebergSnapshotManagement \
   --max-age-hours 0 --min-snapshots 5 --interval 1
mc table warehouse maintenance set aistor/lakehouse --type icebergUnreferencedFileRemoval \
   --unreferenced-days 0 --noncurrent-days 1 --interval 1
```

The values are tuned so the demo finishes in minutes. Production values are in
the README. You can also set the policy in the AIStor console
(http://localhost:9001).

The writer can't change any of this: `ingest-svc` has read-only maintenance
permissions (`s3tables:GetTableMaintenanceConfiguration` and
`s3tables:GetTableMaintenanceJobStatus` in `policies/lakehouse-ingest.json`),
so maintenance stays with the platform team.

## 4. Watch AIStor clean up

Run this cell after setting the policy. It checks every 15 seconds. Nothing in
this notebook touches the table from here on: every change is AIStor working in
the background.

In [ ]:
history = []
objects_at_compaction = None
saw_uncompacted = False
deadline = time.time() + 600
while time.time() < deadline:
    h = lh.health(cat, query_runs=0)
    history.append({"time": pd.Timestamp.now(), **h.as_dict()})
    compacted = h.data_files < before.data_files
    if not compacted:
        saw_uncompacted = True
    elif objects_at_compaction is None:
        objects_at_compaction = h.objects_on_storage
    print(f"{time.strftime('%H:%M:%S')}  data files {h.data_files:>4}  "
          f"snapshots {h.snapshots:>4}  objects on storage {h.objects_on_storage:>5}")
    # Unreferenced file removal has run once objects drop after compaction. If
    # compaction finished before the first check, compare with the start instead.
    if compacted and (h.objects_on_storage < objects_at_compaction or
                      (not saw_uncompacted and h.objects_on_storage < before.objects_on_storage)):
        print("\nCompaction, snapshot expiration and unreferenced file removal have all run.")
        break
    time.sleep(15)
else:
    raise TimeoutError("Maintenance didn't finish within 10 minutes. Run `./scripts/run.sh status` "
                       "in a terminal to see job status before using the after measurements.")


## 5. Before and after

In [ ]:
after = lh.health(cat)
labels = {"rows": "rows", "snapshots": "snapshots", "data_files": "data files (current)",
          "avg_file_kb": "avg data file KB", "objects_on_storage": "objects on storage",
          "storage_mb": "storage MB", "query_ms": "DuckDB query ms (median of 5)"}
b, a = before.as_dict(), after.as_dict()
pd.DataFrame([{"": label, "before": f"{b[k]:,}", "after": f"{a[k]:,}"} for k, label in labels.items()]).set_index("")

In [ ]:
# Four small multiples, each on its own axis, since the measures differ in scale.
BEFORE, AFTER = "#eb6834", "#2a78d6"  # validated pair (colorblind-safe)
panels = [("snapshots", "Snapshots"), ("data_files", "Data files"),
          ("objects_on_storage", "Objects on storage"), ("query_ms", "Query time (ms)")]

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, (key, title) in zip(axes, panels):
    vals = [getattr(before, key), getattr(after, key)]
    bars = ax.bar(["before", "after"], vals, color=[BEFORE, AFTER], width=0.6)
    ax.bar_label(bars, labels=[f"{v:,.0f}" if v >= 10 or float(v).is_integer() else f"{v:,.1f}" for v in vals],
                 padding=3, color="#52514e", fontsize=9)
    ax.set_title(title, fontsize=10, loc="left", color="#0b0b0b")
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.set_yticks([])
    ax.tick_params(axis="x", colors="#52514e", length=0)
    ax.set_ylim(0, max(vals) * 1.18)
fig.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in (BEFORE, AFTER)],
           labels=["before maintenance", "after maintenance"], loc="upper right",
           frameon=False, ncol=2, fontsize=9)
fig.suptitle("iot.sensor_readings: before and after AIStor maintenance", x=0.01, ha="left", fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.9))
plt.show()

## 6. Nothing was lost

Compaction rewrites files, not data. The row count is unchanged, and compaction
shows up in the table history as an Iceberg `replace` snapshot that AIStor
committed.

In [ ]:
table = cat.load_table(lh.IDENTIFIER)
pd.DataFrame([{
    "snapshot_id": s.snapshot_id,
    "committed": pd.to_datetime(s.timestamp_ms, unit="ms"),
    "operation": s.summary.operation.value,
    "added data files": s.summary.get("added-data-files"),
    "deleted data files": s.summary.get("deleted-data-files"),
    "total data files": s.summary.get("total-data-files"),
    "total rows": s.summary.get("total-records"),
} for s in table.snapshots()])

The snapshots kept by `--min-snapshots 5` still work for time travel, so you can
still read the table as it was before compaction. AIStor never deletes a file a
kept snapshot still needs, which is why the original small files stay on storage
until those older snapshots expire.

In [ ]:
oldest = table.snapshots()[0]
print(f"Rows now:                         {table.scan().to_arrow().num_rows:,}")
print(f"Rows at snapshot {oldest.snapshot_id}: {table.scan(snapshot_id=oldest.snapshot_id).to_arrow().num_rows:,}")

## 7. Maintenance metrics

AIStor exports Prometheus-style metrics for every maintenance job, so you can
alert on them like any other part of the platform.

In [ ]:
text = urllib.request.urlopen(f"{lh.AISTOR_URL}/minio/metrics/v3/tables").read().decode()
wanted = ("compaction_files_rewritten_total", "compaction_files_added_total",
          "maintenance_snapshots_expired_total", "maintenance_orphan_files_marked_total")
for line in text.splitlines():
    if line.startswith("minio_tables_") and any(w in line for w in wanted):
        print(line)